Пробовал сделать через tools но падает с 400 ошибкой как понял OpenRouter ограничивает. Пробовал через Siliconflow, выводило что не верный ключ

In [ ]:
import os
import asyncio
import nest_asyncio
from dotenv import load_dotenv
from IPython.display import display, Markdown
from agno.agent import Agent as AgnoAgent
from agno.models.openai import OpenAIChat
from browser_use import Agent as BrowserAgent, ChatBrowserUse
os.environ["ANONYMIZED_TELEMETRY"] = "false"
nest_asyncio.apply()
load_dotenv()

async def run_browser_worker(task_str: str) -> str:
    llm = ChatBrowserUse() 
    agent = BrowserAgent(task=task_str, llm=llm)
    history = await agent.run()
    return history.final_result()

formatter = AgnoAgent(
    model=OpenAIChat(
        id="meta-llama/llama-3.3-70b-instruct:free", 
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        default_headers={"HTTP-Referer": "http://localhost:3000", "X-Title": "Browser Agent"}
    ),
    instructions=[
        "Оформи полученные данные в таблицу Markdown.",api_key=os.getenv("OPENROUTER_API_KEY"),
        default_headers={"HTTP-Referer": "http://localhost:3000", "X-Title": "Browser Agent"}
    ),
        "Столбцы: Название, Цена, Ссылка.",
        "Сделай ссылки кликабельными: [Открыть](URL)."
    ],
    markdown=True,
)

async def main():
    task = "Зайди на Ozon.ru, найди черный чай Greenfield. Выбери первые 3 товара с ценой и ссылкой."
    raw_data = await run_browser_worker(task)
    response = formatter.run(f"Данные: {raw_data}", stream=False)
    display(Markdown(response.content))

await main()

INFO     [Agent] 🔗 Found URL in task: https://Ozon.ru, adding as initial action...
INFO     [Agent] 🎯 Task: Зайди на Ozon.ru, найди черный чай Greenfield. Выбери первые 3 товара с ценой и ссылкой.
INFO     [Agent] Starting a browser-use agent with version 0.11.2, with provider=browser-use and model=bu-1-0
INFO     [Agent]   ▶️   navigate: url: https://Ozon.ru, new_tab: False
INFO     [tools] 🔗 Navigated to https://Ozon.ru
INFO     [Agent] 

INFO     [Agent] 📍 Step 1:
INFO     [Agent]   🧠 Memory: The page is currently loading or displaying a blank screen with only two interactive elements that are likely generic containers. I need to wait for the page to fully load and display the search bar to proceed with searching for "черный чай Greenfield". Since the screenshot is blank, I will wait for a few seconds.
INFO     [Agent]   ▶️   wait: seconds: 3
INFO     [tools] 🕒 waited for 3 seconds
INFO     [Agent] 

INFO     [Agent] 📍 Step 2:
INFO     [Agent]   🧠 Memory: The page has loaded success

| Название | Цена | Ссылка |
| --- | --- | --- |
| Чай в пакетиках черный Greenfield Kenyan Sunrise, 100 шт | 327  | [Открыть](https://www.ozon.ru/product/chay-v-paketikah-chernyy-greenfield-kenyan-sunrise-100-sht-33006462/) |
| Чай в пакетиках черный Greenfield Spring Melody, 100 шт | 335  | [Открыть](https://www.ozon.ru/product/chay-v-paketikah-chernyy-greenfield-spring-melody-100-sht-33006471/) |
| Черный чай в пакетиках Greenfield Golden Ceylon, 100 шт | 308  | [Открыть](https://www.ozon.ru/product/chernyy-chay-v-paketikah-greenfield-golden-ceylon-100-sht-33006463/) |

INFO     [backoff] Backing off send_request(...) for 0.3s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15))
